# Phase 2 - Week 6 - Lesson Day 3
## Evaluation Metrics

Full explanation is in `Day3 Evaluation Metrics.pdf`. All answers are already solved, with comments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

In [ ]:
# Made-up student data: will the student pass (1) or fail (0)?
np.random.seed(42)
n = 300
hours_studied = np.random.uniform(0, 10, n)
attendance_pct = np.random.uniform(50, 100, n)
sleep_hours = np.random.normal(7, 1.2, n)

# The "true" score depends on hours and attendance (sleep does not matter here)
score = hours_studied * 6 + attendance_pct * 0.3 + np.random.normal(0, 8, n)
passed = (score >= 55).astype(int)

df = pd.DataFrame({
    "hours_studied": hours_studied,
    "attendance_pct": attendance_pct,
    "sleep_hours": sleep_hours,
    "passed": passed,
})

X = df[["hours_studied", "attendance_pct", "sleep_hours"]]   # inputs
y = df["passed"]                                              # what we predict

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train the model we will measure in every question below
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
predictions = forest.predict(X_test)

---
## Practice (Solved)

**1. SOLVED - Confusion matrix.** Rows = what really happened, columns = what the model predicted.

In [ ]:
cm = confusion_matrix(y_test, predictions)
print(cm)

tn, fp, fn, tp = cm.ravel()   # unpack the four numbers
print("True Negatives :", tn, "(said Fail, was Fail)")
print("False Positives:", fp, "(said Pass, was Fail)")
print("False Negatives:", fn, "(said Fail, was Pass)")
print("True Positives :", tp, "(said Pass, was Pass)")

**2. SOLVED - The four metrics.**

In [ ]:
print("Accuracy :", round(accuracy_score(y_test, predictions), 3))
print("Precision:", round(precision_score(y_test, predictions), 3))
print("Recall   :", round(recall_score(y_test, predictions), 3))
print("F1       :", round(f1_score(y_test, predictions), 3))

**3. SOLVED - Calculate precision and recall by hand.**

In [ ]:
my_precision = tp / (tp + fp)   # when it says Pass, how often is it right?
my_recall = tp / (tp + fn)      # of the real passes, how many did it find?

print("My precision:", round(my_precision, 3), "| sklearn:", round(precision_score(y_test, predictions), 3))
print("My recall:   ", round(my_recall, 3), "| sklearn:", round(recall_score(y_test, predictions), 3))
# The numbers match, so the formulas are right.

**4. SOLVED - ROC-AUC.** This uses probabilities, not the final yes/no answers.

In [ ]:
probabilities = forest.predict_proba(X_test)[:, 1]   # probability of Pass
print("ROC-AUC:", round(roc_auc_score(y_test, probabilities), 3))
# 0.5 = guessing, 1.0 = perfect. Above 0.8 is usually good.

**5. SOLVED - The accuracy trap.** 100 transactions, only 5 are fraud.

In [ ]:
actual = np.array([1] * 5 + [0] * 95)   # 1 = fraud, 0 = not fraud
lazy_predictions = np.zeros(100, dtype=int)   # a lazy model that ALWAYS says "not fraud"

print("Accuracy:", accuracy_score(actual, lazy_predictions))                  # 0.95
print("Recall:  ", recall_score(actual, lazy_predictions, zero_division=0))   # 0.0
# 95% accuracy sounds great, but this model caught ZERO frauds.
# When one group is rare, always look at precision and recall too.